# AlexNet · CIFAR-100 训练（Colab）

使用前：
1. VS Code 右上角 **Select Kernel** → **Colab** → **GPU (T4)**
2. 运行下面的 GPU 检查 cell，确认 `CUDA available: True`
3. 本地打包代码（PowerShell）：
   ```powershell
   Compress-Archive -Path "D:\DL_Projects\alexnet-cifar100\src","D:\DL_Projects\alexnet-cifar100\requirements.txt" -DestinationPath "D:\alexnet-code.zip" -Force
   ```
4. 把 `alexnet-code.zip` 放到 Google Drive 的 `MyDrive`，或运行后面的上传 cell

训练统一调用 `src/train.py`，不要运行旧版内联模型代码。


In [1]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
!nvidia-smi

PyTorch: 2.14.1+cu130
CUDA available: True
Sun Nov 22 10:13:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+----

## 1. 挂载 Google Drive

最佳权重和 `metrics.json` 会保存到 `/content/drive/MyDrive/alexnet_cifar100`。

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. 准备代码

如果 zip 已经在 Drive 的 `MyDrive` 下，跳过上传 cell，直接运行解压 cell。

如果 zip 还没上传，运行下面的 cell 从本地选择文件。

In [3]:
from pathlib import Path
from google.colab import files

has_zip = Path("/content/drive/MyDrive/alexnet-code.zip").exists() or Path("/content/alexnet-code.zip").exists()
if has_zip:
    print("已找到 alexnet-code.zip，跳过上传")
else:
    uploaded = files.upload()
    print("已上传:", list(uploaded.keys()))

已找到 alexnet-code.zip，跳过上传


In [4]:
from pathlib import Path
import os
import zipfile

PROJECT_DIR = Path("/content/alexnet-cifar100")
candidates = [
    Path("/content/alexnet-code.zip"),
    Path("/content/drive/MyDrive/alexnet-code.zip"),
]
code_zip = next((path for path in candidates if path.exists()), None)

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
if code_zip is not None:
    with zipfile.ZipFile(code_zip) as archive:
        archive.extractall(PROJECT_DIR)
    print("已解压:", code_zip)
elif not (PROJECT_DIR / "src" / "train.py").exists():
    raise FileNotFoundError(
        "没有找到 alexnet-code.zip，请上传到 MyDrive 或 /content 后重跑本 cell"
    )

nested = PROJECT_DIR / "alexnet-cifar100"
if not (PROJECT_DIR / "src" / "train.py").exists() and (nested / "src" / "train.py").exists():
    PROJECT_DIR = nested

os.chdir(PROJECT_DIR)
print("当前目录:", Path.cwd())
!ls src

已解压: /content/drive/MyDrive/alexnet-code.zip
当前目录: /content/alexnet-cifar100
model.py  plot_metrics.py  predict.py  train.py


In [5]:
!pip install -q -r requirements.txt

## 3. 模型自测

应输出 `(2, 4096)`、`(2, 100)` 和参数量 `7,025,316`。

In [6]:
!python src/model.py

特征展平后: torch.Size([2, 4096])
模型输出: torch.Size([2, 100])
参数量: 7025316


## 4. 训练前检查

重新运行训练前先确认没有旧的 `train.py` 进程，否则两个训练会抢同一块 GPU。
如果检查到还有进程，先运行 `!pkill -f src/train.py` 再继续。

In [7]:
!ps -ef | grep "[t]rain.py" || echo "没有正在运行的训练进程"
!du -sh /content/alexnet-cifar100/data 2>/dev/null || echo "数据集还没有下载"
!nvidia-smi

没有正在运行的训练进程
数据集还没有下载
Sun Nov 22 10:14:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------------

## 5. 准备数据集

这个 cell 会显示下载百分比和速度，支持断点续传。
如果已经往 Drive 放了 `cifar-100-python.tar.gz`，会直接使用 Drive 里的压缩包。
下载完成后也会自动把压缩包备份到 Drive，下次内核退出就不用重新下载。

In [8]:
import shutil
import tarfile
import time
from pathlib import Path

import requests

DATA_DIR = Path("/content/alexnet-cifar100/data")
TAR_PATH = DATA_DIR / "cifar-100-python.tar.gz"
DRIVE_TAR = Path("/content/drive/MyDrive/cifar-100-python.tar.gz")
URL = "https://www.cs.toronto.edu/~kriz/cifar-100-python.tar.gz"

DATA_DIR.mkdir(parents=True, exist_ok=True)

if (DATA_DIR / "cifar-100-python" / "train").exists():
    print("CIFAR-100 已经准备好")
else:
    if not TAR_PATH.exists() and DRIVE_TAR.exists():
        TAR_PATH = DRIVE_TAR
        print("使用 Drive 里的压缩包:", TAR_PATH)

    if not TAR_PATH.exists() or TAR_PATH.stat().st_size < 160_000_000:
        start_size = TAR_PATH.stat().st_size if TAR_PATH.exists() else 0
        headers = {"Range": f"bytes={start_size}-"} if start_size else {}
        r = requests.get(URL, headers=headers, stream=True, timeout=60)
        r.raise_for_status()
        mode = "ab" if start_size and r.status_code == 206 else "wb"
        if mode == "wb":
            start_size = 0
        total = int(r.headers.get("Content-Length", 0)) + start_size
        done = start_size
        start_time = time.time()
        last_print = 0
        with open(TAR_PATH, mode) as f:
            for chunk in r.iter_content(chunk_size=512 * 1024):
                if not chunk:
                    continue
                f.write(chunk)
                done += len(chunk)
                if time.time() - last_print > 0.5:
                    last_print = time.time()
                    percent = done / total * 100 if total else 0
                    speed = (done - start_size) / 1024 / 1024 / max(time.time() - start_time, 1e-6)
                    print(
                        f"\r下载中 {percent:5.1f}%  {done/1024/1024:6.1f}/{total/1024/1024:.1f} MB  {speed:.2f} MB/s",
                        end="",
                        flush=True,
                    )
        print()
    else:
        print("压缩包已经完整，跳过下载")

    with tarfile.open(TAR_PATH) as tar:
        tar.extractall(DATA_DIR)
    print("解压完成")
    if TAR_PATH != DRIVE_TAR and not DRIVE_TAR.exists():
        shutil.copy(TAR_PATH, DRIVE_TAR)
        print("压缩包已备份到 Drive:", DRIVE_TAR)

print("数据目录:", DATA_DIR)

下载中  98.3%   158.5/161.2 MB  4.98 MB/s


/tmp/ipykernel_624/2788436382.py:54: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(DATA_DIR)


解压完成
压缩包已备份到 Drive: /content/drive/MyDrive/cifar-100-python.tar.gz
数据目录: /content/alexnet-cifar100/data


## 6. 开始训练

- `EPOCHS` 可以改成 30–50
- 每个 epoch 都会把最佳权重和 `metrics.json` 写入 Drive
- CIFAR-100 会下载到 `/content/alexnet-cifar100/data`，只对本次会话有效
- 如果日志出现 NaN，先停下来把 learning rate 降到 0.005

In [10]:
EPOCHS = 40
BATCH_SIZE = 128
LR = 0.01
OUT_DIR = "/content/drive/MyDrive/alexnet_cifar100"

!python -u src/train.py --epochs {EPOCHS} --batch-size {BATCH_SIZE} --lr {LR} --out-dir {OUT_DIR}

device: cuda
参数量: 7025316
Epoch 01/40 | train_loss 4.0607 | train_acc 0.0713 | test_loss 3.5604 | test_acc 0.1464 | lr 0.010000
  保存新的最好模型，test_acc=0.1464
Epoch 02/40 | train_loss 3.4744 | train_acc 0.1521 | test_loss 3.1024 | test_acc 0.2178 | lr 0.010000
  保存新的最好模型，test_acc=0.2178
Epoch 03/40 | train_loss 3.1307 | train_acc 0.2133 | test_loss 2.7759 | test_acc 0.2803 | lr 0.010000
  保存新的最好模型，test_acc=0.2803
Epoch 04/40 | train_loss 2.8754 | train_acc 0.2622 | test_loss 2.5642 | test_acc 0.3290 | lr 0.010000
  保存新的最好模型，test_acc=0.3290
Epoch 05/40 | train_loss 2.6863 | train_acc 0.3007 | test_loss 2.4840 | test_acc 0.3501 | lr 0.010000
  保存新的最好模型，test_acc=0.3501
Epoch 06/40 | train_loss 2.5218 | train_acc 0.3376 | test_loss 2.2640 | test_acc 0.3944 | lr 0.010000
  保存新的最好模型，test_acc=0.3944
Epoch 07/40 | train_loss 2.4059 | train_acc 0.3640 | test_loss 2.1084 | test_acc 0.4322 | lr 0.010000
  保存新的最好模型，test_acc=0.4322
Epoch 08/40 | train_loss 2.2956 | train_acc 0.3875 | test_loss 2.0772 |

## 7. 绘制训练曲线

生成 `assets/loss_curve.png`、`assets/accuracy_curve.png`、`assets/training_curves.png`。

In [11]:
!python src/plot_metrics.py --metrics /content/drive/MyDrive/alexnet_cifar100/metrics.json

曲线已保存到 assets


## 8. 推理验证

打印 8 个测试样本的真实/预测标签，并保存 2 张正确、1 张错误样例图到 `assets/`。

In [12]:
!python src/predict.py --weights /content/drive/MyDrive/alexnet_cifar100/best_model.pth

checkpoint test_acc: 0.6301
第 0 张 | 真实: mountain | 预测: bridge | 置信度: 0.4946
第 1 张 | 真实: forest | 预测: forest | 置信度: 0.5409
第 2 张 | 真实: seal | 预测: otter | 置信度: 0.2515
第 3 张 | 真实: mushroom | 预测: mushroom | 置信度: 0.8084
第 4 张 | 真实: sea | 预测: sea | 置信度: 0.8153
第 5 张 | 真实: tulip | 预测: spider | 置信度: 0.7814
第 6 张 | 真实: camel | 预测: tractor | 置信度: 0.2758
第 7 张 | 真实: butterfly | 预测: raccoon | 置信度: 0.2172
保存图片: assets/prediction_correct_1.png
保存图片: assets/prediction_correct_2.png
保存图片: assets/prediction_wrong_1.png


## 9. 收尾

1. 确认 Drive 里的 `alexnet_cifar100/` 有 `best_model.pth` 和 `metrics.json`
2. 把训练得到的 `assets/` 图片和 `metrics.json` 提交到 GitHub
3. 把 `best_model.pth` 上传到 Hugging Face Model 仓库
4. 更新 README 中的准确率和链接

如果 Drive 没有自动同步到本地，可以运行下面的 cell 直接下载权重。

In [13]:
from google.colab import files
files.download("/content/drive/MyDrive/alexnet_cifar100/best_model.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
!ls -lh /content/drive/MyDrive/alexnet_cifar100


total 54M
-rw------- 1 root root  27M Nov 22 10:34 best_model.pth
-rw------- 1 root root  27M Nov 22 10:35 last_model.pth
-rw------- 1 root root 7.3K Nov 22 10:35 metrics.json


In [15]:
!python -c "import json; d=json.load(open('/content/drive/MyDrive/alexnet_cifar100/metrics.json')); print('epochs:', len(d['history'])); print('best:', max(x['test_acc'] for x in d['history']))"


epochs: 40
best: 0.6301


In [16]:
!mkdir -p /content/drive/MyDrive/alexnet_cifar100/assets
!cp /content/alexnet-cifar100/assets/*.png /content/drive/MyDrive/alexnet_cifar100/assets/
!ls -lh /content/drive/MyDrive/alexnet_cifar100/assets


total 220K
-rw------- 1 root root 51K Nov 22 10:38 accuracy_curve.png
-rw------- 1 root root 50K Nov 22 10:38 loss_curve.png
-rw------- 1 root root 14K Nov 22 10:38 prediction_correct_1.png
-rw------- 1 root root 15K Nov 22 10:38 prediction_correct_2.png
-rw------- 1 root root 14K Nov 22 10:38 prediction_wrong_1.png
-rw------- 1 root root 76K Nov 22 10:38 training_curves.png
